# Lab 6 · Dữ liệu ngoài: đọc đúng kiểu, Parquet, API lịch sử và DuckDB

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 6**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng Bài 6 dùng bảng năm hàng giả lập và API dự báo thời tiết. Lab này làm các việc đó trên
dữ liệu thật của Santiago: bảng đầy đủ 90 cột, nhiệt độ quá khứ từ API lịch sử và 690.112 đánh giá (review),
rồi đối chiếu kết quả với lab 2.

## Cách làm việc trong bài lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  trên lớp (🚫 đóng) đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau bài lab, bạn sẽ:

1. Khai báo kiểu và giá trị thiếu khi đọc CSV; đọc có chọn lọc bảng 90 cột và so sánh bộ nhớ.
2. Lưu bảng bằng Parquet và kiểm tra kiểu dữ liệu sau khi đọc lại, so với CSV.
3. Chọn tham số cho API lịch sử, kiểm tra đơn vị, lưu phản hồi thô rồi ghép với số đánh giá theo ngày.
4. Viết truy vấn SQL với DuckDB (`GROUP BY`, `COUNT`, `LEFT JOIN`, `HAVING`) và kiểm chứng bằng con số
   đã biết hoặc bằng pandas.

In [ ]:
%pip install -q "duckdb==1.5.5"

## Phần 0 · Khởi động (~10 phút)

In [ ]:
import pandas as pd
from pathlib import Path

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

# W1 — khai báo kiểu và giá trị thiếu khi đọc, như bảng năm hàng ở bài giảng
Path("data/raw/w1.csv").write_text(
    "ma,khu,gia\n013,Santiago,45000\n021,Ñuñoa,-\n101,Providencia,52000\n",
    encoding="utf-8",
)
# TODO: đọc data/raw/w1.csv sao cho cột ma giữ đủ 3 ký tự
#       và dấu "-" ở cột gia được hiểu là giá trị thiếu
bang = ...

# --- Ô kiểm tra ---
assert bang["ma"].tolist() == ["013", "021", "101"]
assert bang["gia"].isna().sum() == 1 and pd.api.types.is_numeric_dtype(bang["gia"])
print("W1 ổn: mã giữ số 0 ở đầu, dấu '-' thành giá trị thiếu, cột gia là số.")

In [ ]:
# W2 — đọc 690.112 đánh giá: listing_id là mã nên đọc thành chuỗi;
#      parse_dates chuyển cột date sang kiểu ngày giờ ngay khi đọc (xem phần tự học của notebook bài giảng)
URL_RV = ("https://data.insideairbnb.com/chile/rm/santiago/"
          "2026-06-29/visualisations/reviews.csv")
# TODO: đọc URL_RV vào biến rv, khai báo kiểu cho listing_id và dùng parse_dates=["date"]
rv = ...

# --- Ô kiểm tra ---
assert len(rv) == 690112
assert pd.api.types.is_string_dtype(rv["listing_id"])
assert str(rv["date"].dtype).startswith("datetime64")
print("W2 ổn: 690.112 đánh giá, listing_id là chuỗi, date là kiểu ngày giờ.")

## Phần 1 · Bảng đầy đủ 90 cột: đọc có chọn lọc và lưu Parquet (~25 phút)

Các bài trước dùng bản rút gọn 19 cột. Bảng **đầy đủ** của Inside Airbnb có 90 cột và được nén
(`.csv.gz`); pandas tự giải nén khi đọc. Ô dưới tải file một lần vào `data/raw/`, các lần chạy sau dùng lại.

In [ ]:
import requests

URL_FULL = ("https://data.insideairbnb.com/chile/rm/santiago/"
            "2026-06-29/data/listings.csv.gz")
TEP_FULL = Path("data/raw/listings_full_2026-06-29.csv.gz")
if not TEP_FULL.exists():
    r_full = requests.get(URL_FULL, timeout=120)
    r_full.raise_for_status()
    TEP_FULL.write_bytes(r_full.content)
print(f"{TEP_FULL}: {TEP_FULL.stat().st_size / 1e6:.1f} MB")

### Bước 1 · Đọc cả bảng và chỉ đọc 5 cột

Các bước sau chỉ cần 5 cột. Trước khi chạy, dự đoán: bảng 5 cột nhẹ hơn bảng 90 cột bao nhiêu lần về bộ nhớ,
và đọc nhanh hơn bao nhiêu lần?

In [ ]:
import time


def mb(df):
    """Bộ nhớ của một DataFrame, tính bằng MB."""
    return df.memory_usage(deep=True).sum() / 1e6


# Cách 1: đọc cả bảng — ô này cho sẵn
t0 = time.perf_counter()
ca_bang = pd.read_csv(TEP_FULL)
t_ca_bang = time.perf_counter() - t0

# TODO Cách 2: chỉ đọc 5 cột ["id", "room_type", "price", "first_review", "number_of_reviews"];
#      id là mã nên đọc thành chuỗi; first_review chuyển sang kiểu ngày giờ bằng parse_dates
t0 = time.perf_counter()
nho = ...
t_nho = time.perf_counter() - t0

print(f"Cả bảng: {ca_bang.shape}, {mb(ca_bang):.1f} MB, {t_ca_bang:.2f} s")
print(f"5 cột  : {nho.shape}, {mb(nho):.1f} MB, {t_nho:.2f} s")

# --- Ô kiểm tra ---
assert ca_bang.shape == (18534, 90) and nho.shape == (18534, 5)
assert pd.api.types.is_string_dtype(nho["id"])
assert str(nho["first_review"].dtype).startswith("datetime64")
assert mb(nho) < mb(ca_bang) / 10
print("Cột price:", nho["price"].dropna().head(3).tolist())

Bộ nhớ giảm hàng chục lần, còn thời gian giảm ít hơn nhiều: với CSV, pandas vẫn phải giải nén và quét cả file
văn bản; `usecols` chỉ bỏ bước chuyển đổi các cột không dùng. Cột `price` của bảng đầy đủ là chuỗi dạng
`"$45,647.00"`, chưa tính toán được; xử lý chuỗi là nội dung của Bài 7.

### Bước 2 · Lưu CSV và Parquet, đọc lại và so kiểu cột

Dự đoán: ghi bảng `nho` ra CSV rồi đọc lại (không khai báo gì), cột `id` và `first_review` còn giữ kiểu không?
Với Parquet thì sao?

In [ ]:
nho.to_csv("data/processed/listings_5cot.csv", index=False)
nho.to_parquet("data/processed/listings_5cot.parquet", index=False)

# TODO: đọc lại hai file (không khai báo kiểu) vào lai_csv, lai_pq
#       và tính kích thước hai file theo MB (gợi ý: Path(...).stat().st_size)
lai_csv = ...
lai_pq = ...
mb_csv = ...
mb_pq = ...

print(f"CSV {mb_csv:.2f} MB — Parquet {mb_pq:.2f} MB")
print("Đọc lại từ CSV    :", lai_csv["id"].dtype, "|", lai_csv["first_review"].dtype)
print("Đọc lại từ Parquet:", lai_pq["id"].dtype, "|", lai_pq["first_review"].dtype)

# --- Ô kiểm tra ---
assert pd.api.types.is_integer_dtype(lai_csv["id"])
assert not str(lai_csv["first_review"].dtype).startswith("datetime64")
assert (lai_pq.dtypes == nho.dtypes).all()   # Parquet giữ nguyên kiểu từng cột
assert lai_pq["id"].equals(nho["id"])
assert mb_pq < mb_csv

CSV chỉ lưu văn bản: đọc lại thì mã thành số, ngày thành chuỗi, phải khai báo lại như ở Bước 1. Parquet lưu
kiểu cùng dữ liệu nên đọc lại được đúng bảng. Với bảng này, file Parquet còn nhỏ hơn khoảng 2 lần, nhưng tỷ lệ
còn tuỳ dữ liệu. Phần 3 truy vấn thẳng file Parquet này.

## Phần 2 · API lịch sử và số đánh giá theo ngày (~30 phút)

Câu hỏi: *trong tháng 1/2025, ngày nóng hơn có nhiều đánh giá hơn không?* Cần hai nguồn: nhiệt độ quá khứ từ
API lịch sử (archive) của Open-Meteo và số đánh giá theo ngày từ `rv`. Dùng một khoảng thời gian cố định trong
quá khứ giúp phân tích chạy lại được cho cùng kết quả.

### Bước 3 · Chọn tham số và gọi API

Tra [tài liệu API lịch sử của Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api) để điền các
tham số còn thiếu: nhiệt độ tối đa từng ngày, từ 01/01/2025 đến 31/01/2025, ngày tính theo giờ Santiago.

In [ ]:
import json

tham_so = {
    "latitude": -33.45,
    "longitude": -70.66,
    # TODO: thêm 4 tham số: ngày bắt đầu, ngày kết thúc, đại lượng theo ngày, múi giờ
}
r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=tham_so, timeout=30)
r.raise_for_status()
du_lieu = r.json()
print(r.url)

# --- Ô kiểm tra ---
assert du_lieu["timezone"] == "America/Santiago", "Kiểm tra tham số múi giờ"
assert "daily" in du_lieu, "Thiếu đại lượng theo ngày"
assert du_lieu["daily"]["time"][0] == "2025-01-01" and len(du_lieu["daily"]["time"]) == 31, "Kiểm tra khoảng ngày"

In [ ]:
# Lưu nguyên phản hồi (r.content) kèm URL và thời điểm tải, như ở bài giảng
tep_tt = Path("data/raw/thoi_tiet_santiago_2025-01.json")
tep_tt.write_bytes(r.content)
tep_tt.with_suffix(".meta.json").write_text(
    json.dumps({"url": r.url, "tai_luc_utc": pd.Timestamp.now(tz="UTC").isoformat()}, indent=2),
    encoding="utf-8",
)
print("Đã lưu:", tep_tt)

### Bước 4 · Tạo bảng theo ngày và kiểm tra đơn vị

In [ ]:
# TODO: tạo bảng thoi_tiet từ phần "daily" của du_lieu, mỗi hàng một ngày
thoi_tiet = ...

# --- Ô kiểm tra ---
assert du_lieu["daily_units"]["temperature_2m_max"] == "°C"
assert len(thoi_tiet) == 31 and list(thoi_tiet.columns) == ["time", "temperature_2m_max"]
tb_max = thoi_tiet["temperature_2m_max"].mean()
assert 27 <= tb_max <= 34, f"Nhiệt độ cao nhất trung bình tháng 1 ở Santiago khoảng 30–31°C, đang ra {tb_max:.1f}"
print(f"31 ngày, nhiệt độ cao nhất trung bình {tb_max:.1f}°C: đúng mùa hè ở bán cầu nam.")

### Bước 5 · Đếm đánh giá theo ngày

In [ ]:
# TODO: đếm số đánh giá theo từng ngày của tháng 1/2025 từ rv
#   1) lọc rv trong khoảng 2025-01-01 .. 2025-01-31 vào jan
#   2) groupby theo chuỗi ngày jan["date"].dt.strftime("%Y-%m-%d") rồi size()
#      (strftime đổi ngày thành chuỗi, Bài 8 sẽ học kỹ)
#   3) đưa về DataFrame 2 cột: time, so_review (reset_index + đặt tên)
jan = ...
theo_ngay = ...

# --- Ô kiểm tra ---
assert len(jan) == 14636 and len(theo_ngay) == 31
assert list(theo_ngay.columns) == ["time", "so_review"]
theo_ngay.head(3)

### Bước 6 · Ghép hai nguồn và đo tương quan

In [ ]:
# TODO: merge thoi_tiet với theo_ngay theo cột "time" (how="left")
gop = ...

# --- Ô kiểm tra ---
assert gop.shape == (31, 3) and gop["so_review"].isna().sum() == 0

# Tương quan giữa nhiệt độ tối đa và số đánh giá theo ngày
he_so = gop["temperature_2m_max"].corr(gop["so_review"])
print(f"Hệ số tương quan: {he_so:.3f}")

Hệ số khoảng 0,08 cho thấy mối liên hệ tuyến tính giữa nhiệt độ tối đa và số đánh giá theo ngày trong mẫu
này rất yếu. Kết quả không ủng hộ giả thuyết ngày nóng hơn có nhiều đánh giá hơn. Khi diễn giải, cần lưu ý số
đánh giá còn phụ thuộc lịch trả phòng và thời điểm khách viết đánh giá.

## Phần 3 · SQL với DuckDB (~30 phút)

DuckDB tìm DataFrame theo tên biến (`FROM rv`) và đọc thẳng file Parquet đã lưu ở Phần 1. Dùng
`duckdb.sql(...).df()` như bài giảng; hàm `year(date)` lấy năm từ một ngày.

### Bước 7 · Đếm đánh giá theo năm và đối chiếu với lab 2

In [ ]:
import duckdb

# TODO: đếm số đánh giá theo năm từ DataFrame rv
#   SELECT year(date) AS nam, COUNT(*) AS n FROM rv GROUP BY ... ORDER BY nam
dem_nam = duckdb.sql(...).df()

# --- Ô kiểm tra: đối chiếu với con số đếm bằng Python thuần ở lab 2 ---
nam_2025 = int(dem_nam.loc[dem_nam["nam"] == 2025, "n"].iloc[0])
assert nam_2025 == 211432, "Phải khớp đúng con số đếm bằng dict ở lab 2!"
print("SQL và Python thuần, hai công cụ độc lập, cùng một đáp số: 211.432 đánh giá năm 2025.")

### Bước 8 · `COUNT(*)` và `COUNT(cột)` trên file Parquet

Chỗ ở chưa có đánh giá nào thì `first_review` bị thiếu. Với mỗi loại chỗ ở, đếm tổng số chỗ ở và số chỗ ở đã
có ít nhất một đánh giá.

In [ ]:
# TODO: đọc thẳng file 'data/processed/listings_5cot.parquet'; với mỗi room_type, đếm
#       COUNT(*) AS so_cho_o và COUNT(first_review) AS so_co_danh_gia; sắp giảm dần theo so_cho_o
theo_loai_cho = duckdb.sql(...).df()

# --- Ô kiểm tra ---
assert theo_loai_cho["so_cho_o"].sum() == 18534
assert theo_loai_cho["so_co_danh_gia"].sum() == (nho["number_of_reviews"] > 0).sum()
theo_loai_cho

### Bước 9 · Ghép đánh giá với loại chỗ ở bằng `LEFT JOIN`

`listing_id` trong `rv` và `id` trong file Parquet đều là chuỗi (W2, Bước 1), nên ghép trực tiếp được.
`LEFT JOIN` giữ mọi đánh giá; đánh giá nào không tìm được chỗ ở sẽ có `room_type` thiếu.

In [ ]:
# TODO: đếm số đánh giá năm 2025 theo loại chỗ ở
#   FROM rv LEFT JOIN 'data/processed/listings_5cot.parquet' AS l ON rv.listing_id = l.id
#   WHERE year(rv.date) = 2025; GROUP BY room_type; COUNT(*) AS n; sắp giảm dần theo n
theo_loai = duckdb.sql(...).df()

# --- Ô kiểm tra ---
assert theo_loai["n"].sum() == 211432               # không mất đánh giá nào của năm 2025
assert theo_loai["room_type"].isna().sum() == 0     # đánh giá nào cũng tìm được chỗ ở
assert theo_loai.iloc[0]["room_type"] == "Entire home/apt" and int(theo_loai.iloc[0]["n"]) == 192326
theo_loai

Nhà nguyên căn nhận 192.326 trên 211.432 đánh giá năm 2025 (khoảng 91%), cao hơn tỷ trọng của nhóm này trong
số chỗ ở (khoảng 81%).

### Bước 10 · Lọc nhóm bằng `HAVING`

Có bao nhiêu chỗ ở nhận từ 100 đánh giá trở lên trong năm 2025? `WHERE` lọc từng đánh giá theo năm, `HAVING`
lọc từng chỗ ở theo số đánh giá.

In [ ]:
# TODO: mỗi hàng kết quả là một chỗ ở có từ 100 đánh giá trở lên trong năm 2025
#   SELECT listing_id, COUNT(*) AS n FROM rv WHERE ... GROUP BY listing_id HAVING ...
nhieu_danh_gia = duckdb.sql(...).df()

# --- Ô kiểm tra: đối chiếu bằng pandas ---
dem_pd = rv.loc[rv["date"].dt.year == 2025, "listing_id"].value_counts()
assert len(nhieu_danh_gia) == (dem_pd >= 100).sum() == 90
print(len(nhieu_danh_gia), "chỗ ở có từ 100 đánh giá trở lên trong năm 2025")

## Phần 4 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt và cách kiểm chứng.

### Tự làm 1 · So sánh với tháng 7

Lặp Phần 2 cho **tháng 7/2025** (giữa mùa đông): gọi API lịch sử, đếm đánh giá theo ngày, ghép, tính tương
quan. So với tháng 1 và viết 2 câu nhận xét. Lưu phản hồi thô vào `data/raw/`.

### Tự làm 2 · Chỗ ở có nhiều đánh giá nhất

Viết truy vấn DuckDB tìm **5 chỗ ở có nhiều đánh giá nhất năm 2025**, kèm tên (cột `name` của `ca_bang`). Chú ý
kiểu của hai cột khoá: `listing_id` là chuỗi, còn `id` của `ca_bang` là số. Kiểm chứng con số đầu bảng bằng
pandas (`rv`, lọc năm, `value_counts` trên `listing_id`).

In [ ]:
# Viết bài tự làm của bạn ở đây

## Tóm tắt bài lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Khai báo kiểu, giá trị thiếu; `usecols` trên bảng 90 cột | đọc file nhiều cột một cách có chọn lọc |
| CSV làm mất kiểu, Parquet giữ kiểu | dữ liệu trung gian trong `data/processed/` |
| Tham số API, đơn vị, lưu phản hồi thô | gọi API ở Bài 11 |
| Ghép hai nguồn theo ngày, diễn giải tương quan yếu | thẩm định kết luận ở Bài 14 |
| DuckDB: `COUNT`, `LEFT JOIN`, `HAVING`, đối chiếu với lab 2 và pandas | kiểm chứng chéo bằng hai công cụ |

Bài giảng tiếp theo: xử lý **chuỗi và biểu thức chính quy (regex)**, gồm cả cột giá dạng `"$45,647.00"`.